## Taonaishe Hastings Muzavazi R2420726  
## HASTS211 Assignment 

### Identifying and Detecting a Regime Change in the AAPL Stock Returns from yahoo finance 

This notebook applies the Hidden Markov Models to detect the change in the regime 

Source of data: Yahoo Finance AAPL data 
Dataset : AAPL data from 1 January 2018 to 31 December 2025 

---
### 1. Definition

### Regime Detection 
A regime change is a shift in market behavior. Or a change in the behaviour of a given time series at a given time interval.
We can then say the time series has several possible states rather than one.

### Definition of the Hidden Markov Model 
is a Markov model in which the observations are dependent on a latent (hidden) Markov process X. The observable process Y whose outcomes depend on the outcomes of X in a known way. 

Formular definition: 
Let X_n and Y_n be discrete-time stochastic processes and n >= 1. The pair (X_n,Y_n) is a hidden Markov model if
X_n is a Markov process whose behavior is not directly observable 
$$ P(Y_n = i | X_1=x_1,...,X_n=x_n) = P(Y_n = i| X_n=x_n) $$

Y in this case is the returns or the prices. 
X will be the hidden state of the market (either calm or turbulent)

#### Why the Hidden Markov Model?
 The model is well suited because it involves the inference on "hidden" generative processes via "noisy" indirect observations correlated to the processes. 
In this case the hidden or latent process is the underlying regime state while asset returns are the noisy indrect observations.
The Model is assuming that there are 2 hidden market regimes: a low-volatility (stable /bull market) and a high-volatility market (unstable / bear market). The main goal is to determine the underlying hidden state given the observed data (returns or market prices.




### Installing the necessary modules and packages 

In this notebook plotly will be used instead of matplotlib so that we can be able to zoom in on our plots and reduce or increase the time (for example we can zoom in until we are able to see the daily plot rather than the fixed years.)

In [18]:
!pip install yfinance scikit-learn plotly hmmlearn statsmodels

In [19]:
import yfinance as yf 
import numpy as np 
import pandas as pd 
from hmmlearn.hmm import GaussianHMM 
import plotly.graph_objects as go 
from plotly.graph_objs.scatter.marker import Line 
from plotly.subplots import make_subplots 
import plotly.express as px 
from sklearn.cluster import AgglomerativeClustering 
from sklearn.mixture import GaussianMixture 
from statsmodels.tsa.stattools import adfuller 
from scipy import stats 
import warnings 
import math 

warnings.filterwarnings('ignore')

### Loading the Data 
We are going to use yfinance module to download the finance data from yahoo finance for Apple Inc. 



In [20]:
#Loading the Apple data 
apple_data = yf.download("AAPL",start='2018-01-01',end='2025-12-31',auto_adjust=True)
apple = pd.DataFrame(apple_data)
apple.columnns = apple_data['Close']

#computing log_returns 
apple['log_returns']= np.log(apple['Close']/apple['Close'].shift(1))
apple = apple.dropna()
print(f'Observations: {len(apple)}')
print(f'Dates : {apple.index[0].date()} to {apple.index[-1].date()}')
apple.head()

[*********************100%***********************]  1 of 1 completed

Observations: 2009
Dates : 2018-01-03 to 2025-12-30


Price,Close,High,Low,Open,Volume,log_returns
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,
Date,,,,,,
2018-01-03,40.297157,40.839976,40.233987,40.367350,118071600,-0.000174
2018-01-04,40.484333,40.587282,40.262059,40.369685,89738400,0.004634
2018-01-05,40.945248,41.031816,40.489001,40.580251,94640000,0.011321
2018-01-08,40.793175,41.087979,40.694903,40.793175,82271200,-0.003721
2018-01-09,40.788498,40.959297,40.573243,40.839972,86336000,-0.000115


The missing values must have been left out by the drop.na() functions, which means they would have produced null values which we should drop for proper data analysis.

---
## 3. Diagram — Exploratory Plots

Observing the plot of the data in order to discover any visible trends before we fit the model 

From just looking at the graph above it can be observed that there is a break in the upward trend that is around 2020, most of 2022 and then also in 2025. 

In [21]:
# Rolling 30-day annualised volatility
rolling_vol = apple['log_returns'].rolling(30).std() * np.sqrt(252)

fig = make_subplots(rows=3, cols=1,
    subplot_titles=['AAPL Adjusted Close Price (2018–2025)',
                    'AAPL Daily Log Returns',
                    'AAPL 30-Day Rolling Annualised Volatility'],
    shared_xaxes=True, vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=apple.index, y=apple_data['Close'].squeeze(),
    line=dict(color='green'), name='Close Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=apple.index, y=apple['log_returns'],
    line=dict(color='blue', width=0.8), name='Log Return'), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=0.8, row=2, col=1)

fig.add_trace(go.Scatter(x=apple.index, y=rolling_vol,
    line=dict(color='orange'), name='Rolling Vol'), row=3, col=1)

fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
fig.update_yaxes(title_text='Log Return', row=2, col=1)
fig.update_yaxes(title_text='Ann. Volatility', row=3, col=1)
fig.update_layout(height=800, width=900, showlegend=False, margin=dict(l=20,r=20,t=60,b=20))
fig.show()
print('From just looking at the graph above it can be observed that there is a break in the upward trend that is around 2020 (COVID19), most of 2022 (Geopolitics(Russian Urkraine war)) and then also in 2025(geopolitics shock and change in USA administration). ')

From just looking at the graph above it can be observed that there is a break in the upward trend that is around 2020 (COVID19), most of 2022 (Geopolitics(Russian Urkraine war)) and then also in 2025(geopolitics shock and change in USA administration). 


In [22]:
# Return distribution: histogram vs normal + QQ plot
lr = apple['log_returns'].dropna()
mu, sigma = lr.mean(), lr.std()
x_range = np.linspace(lr.min(), lr.max(), 200)

# QQ data
(osm, osr), (slope, intercept, _) = stats.probplot(lr, dist='norm')
qq_line_x = np.array([osm[0], osm[-1]])
qq_line_y = intercept + slope * qq_line_x

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Return Distribution vs Normal', 'QQ Plot of AAPL Log Returns'])

fig.add_trace(go.Histogram(x=lr, nbinsx=70, histnorm='probability density',
    marker_color='steelblue', opacity=0.6, name='AAPL returns'), row=1, col=1)
fig.add_trace(go.Scatter(x=x_range, y=stats.norm.pdf(x_range, mu, sigma),
    line=dict(color='red', width=2), name='Normal'), row=1, col=1)

fig.add_trace(go.Scatter(x=osm, y=osr, mode='markers',
    marker=dict(color='steelblue', size=3), name='QQ'), row=1, col=2)
fig.add_trace(go.Scatter(x=qq_line_x, y=qq_line_y,
    line=dict(color='red'), name='QQ line'), row=1, col=2)

fig.update_xaxes(title_text='Log Return', row=1, col=1)
fig.update_yaxes(title_text='Density', row=1, col=1)
fig.update_xaxes(title_text='Theoretical Quantiles', row=1, col=2)
fig.update_yaxes(title_text='Sample Quantiles', row=1, col=2)
fig.update_layout(height=450, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()
print('The heavy tails indicate that ')

The heavy tails indicate that 


---
## 4. Demonstration — Fitting the Model

We fit a 2-state Gaussian HMM to the AAPL log returns. The model will try to separate the data into a low-volatility regime and a high-volatility regime.

In order to fit our model we define a python object named RegimeDetection. We then encapsulate a function to determine the market regime using HMM models. 
We then define initialise_model function, which acts as a constructor of the model as an attribute. 


In [23]:
class RegimeDetection:
    def get_regimes_hmm(self, input_data, params):
        hmm_model = self.initialise_model(GaussianHMM(),params).fit(input_data)
        return hmm_model 
    
    def initialise_model(self,model,params): 
        for parameter, value in params.items():
            setattr(model,parameter,value)
        return model 
    

Further more we define a separate function to plot the returned hidden states along with the close prices. 

In [24]:
def plot_hidden_states(hidden_states, prices_df): 
    '''
    Input: 
    hidden_states(numpy.ndarray) -  array of predicted hidden states
    prices_df - dataframe of close prices 

    Output: 
    Graph showing hidden states and prices 
    '''
    colors = ['blue','green']
    n_components = len(np.unique(hidden_states))
    fig = go.Figure()

    for i in range(n_components):
        mask = hidden_states == i 
        print(f'Number of observations for State {i} : {len(prices_df.index[mask])}')

        fig.add_trace(go.Scatter(x=prices_df.index[mask],y=prices_df[mask],
                    mode = 'markers', name= 'Hidden State' + str(i), marker = dict(size=4, color=colors[i]) ))
    fig.update_layout(height=400, width = 900, legend = dict(yanchor="top", y=0.99, xanchor="left",x=0.01),
                      margin=dict(l=20,r=20,t=20,b=20)).show()


After this we can then initialize the RegimeDetection object

In [25]:
regime_detection = RegimeDetection()

In [26]:
# making an array for the prices to be used in the model 
price_array = np.array([[q] for q in apple['log_returns'].values])

In [27]:
params = {'n_components':2, 'covariance_type':'full','random_state':100}
hmm_model = regime_detection.get_regimes_hmm(price_array, params)
hmm_states = hmm_model.predict(price_array)
plot_hidden_states(np.array(hmm_states),apple['log_returns'])

Number of observations for State 0 : 196
Number of observations for State 1 : 1813


In [28]:
# Extract and display model parameters
means = hmm_model.means_.flatten()
vols  = np.sqrt(hmm_model.covars_.flatten())

# Sort so state 0 = bull (higher mean) and state 1 = bear (lower mean)
order = np.argsort(means)[::-1]
means = means[order]
vols  = vols[order]
trans = hmm_model.transmat_[np.ix_(order, order)]
labels = ['Bull (low volatility)', 'Bear (high volatility)']

print('Estimated Model Parameters')
print('='*45)
for i in range(2):
    print(f'\nRegime {i+1}: {labels[i]}')
    print(f'  Daily mean return:     {means[i]*100:.4f}%')
    print(f'  Annualised return:     {means[i]*252*100:.2f}%')
    print(f'  Daily volatility:      {vols[i]*100:.4f}%')
    print(f'  Annualised volatility: {vols[i]*np.sqrt(252)*100:.2f}%')

print('\nTransition Probability Matrix')
trans_df = pd.DataFrame(trans,
    index=[f'From {l}' for l in labels],
    columns=[f'To {l}' for l in labels])
print(trans_df.round(4))

print('\nExpected number of days in each regime before switching:')
for i in range(2):
    duration = 1 / (1 - trans[i, i])
    print(f'  {labels[i]}: {duration:.1f} days (~{duration/21:.1f} months)')

Estimated Model Parameters

Regime 1: Bull (low volatility)
  Daily mean return:     0.1599%
  Annualised return:     40.30%
  Daily volatility:      1.4356%
  Annualised volatility: 22.79%

Regime 2: Bear (high volatility)
  Daily mean return:     -0.3804%
  Annualised return:     -95.85%
  Daily volatility:      4.0943%
  Annualised volatility: 64.99%

Transition Probability Matrix
                             To Bull (low volatility)  \
From Bull (low volatility)                     0.9644   
From Bear (high volatility)                    0.2611   

                             To Bear (high volatility)  
From Bull (low volatility)                      0.0356  
From Bear (high volatility)                     0.7389  

Expected number of days in each regime before switching:
  Bull (low volatility): 28.1 days (~1.3 months)
  Bear (high volatility): 3.8 days (~0.2 months)


In [29]:
# Regime decoding with posterior probabilities
remap = {order[0]: 0, order[1]: 1}
hmm_states_sorted = np.array([remap[s] for s in hmm_states])

apple['Regime'] = hmm_states_sorted

posteriors = hmm_model.predict_proba(price_array)
apple['P_Bull'] = posteriors[:, order[0]]
apple['P_Bear'] = posteriors[:, order[1]]

print('Days in each regime:')
counts = apple['Regime'].value_counts().sort_index()
for i, lbl in enumerate(labels):
    pct = counts[i] / len(apple) * 100
    print(f'  {lbl}: {counts[i]} days ({pct:.1f}%)')

Days in each regime:
  Bull (low volatility): 1813 days (90.2%)
  Bear (high volatility): 196 days (9.8%)


In [30]:
# Regime-coloured plots: price, returns, posterior probabilities
close_series = apple_data['Close'].squeeze().loc[apple.index]

fig = make_subplots(rows=3, cols=1,
    subplot_titles=['AAPL Price — Coloured by Detected Regime',
                    'AAPL Log Returns — Coloured by Detected Regime',
                    'Probability of Each Regime Over Time'],
    shared_xaxes=True, vertical_spacing=0.07)

colors_regime = {0: 'green', 1: 'blue'}
for regime in [0, 1]:
    mask = apple['Regime'] == regime
    fig.add_trace(go.Scatter(
        x=apple.index[mask], y=close_series[mask],
        mode='markers', marker=dict(size=3, color=colors_regime[regime]),
        name=labels[regime], legendgroup=labels[regime],
        showlegend=True), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=apple.index[mask], y=apple['log_returns'][mask],
        mode='markers', marker=dict(size=3, color=colors_regime[regime]),
        name=labels[regime], legendgroup=labels[regime],
        showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(x=apple.index, y=apple['P_Bull'],
    fill='tozeroy', fillcolor='rgba(0,128,0,0.3)',
    line=dict(color='green'), name='P(Bull)'), row=3, col=1)
fig.add_trace(go.Scatter(x=apple.index, y=apple['P_Bear'],
    fill='tozeroy', fillcolor='rgba(173,216,230,0.3)',
    line=dict(color='blue'), name='P(Bear)'), row=3, col=1)

fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=0.6, row=2, col=1)
fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
fig.update_yaxes(title_text='Log Return', row=2, col=1)
fig.update_yaxes(title_text='Probability', row=3, col=1)
fig.update_layout(height=900, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()

### Interpretation 

From the model output above:

- **Bear regime**: This state is characterized by high volatility and low prices. This is mostly observed in the year 2020 around Feb to May, 2022 as well as 2025. After each period of the bear regime the market finds its way back to the normal (bull regime). 

- **Bull regime**: During these times there are increasing prices and low volatility (also low variance of the data). This is observed for majority of the time in years such as 2018 most of 2019, 2023 and 2024.


- **Probability of regime** : From the graph, majority of the probability for bullish regime are higher than the probabilities for the bearish regime. This indicates that the bullish market is more dominant and to be expected most of the times.

---
## 5. Diagnosis — Checking the Model

In [31]:
# ADF test — check that log returns are stationary
adf_stat, p_val, lags, nobs, crit, _ = adfuller(apple['log_returns'], autolag='AIC')

print('ADF Test on AAPL Log Returns')
print(f'ADF statistic: {adf_stat:.4f}')
print(f'p-value:       {p_val:.6f}')
print(f'Critical values: {crit}')
print('')
if p_val < 0.05:
    print('Result: reject the unit root meaning that the log returns are stationary')
    print('This confirms the regime switching is in terms of variance not the level')
else:
    print('Result: cannot reject unit root')

ADF Test on AAPL Log Returns
ADF statistic: -14.7011
p-value:       0.000000
Critical values: {'1%': np.float64(-3.433623856429125), '5%': np.float64(-2.862986213505), '10%': np.float64(-2.56753990225)}

Result: reject the unit root meaning that the log returns are stationary
This confirms the regime switching is in terms of variance not the level


Since we are using a Gaussian model we would like to check whether the data indeed follows a normal distribution. 
This can be done by comparing the standardized residuals to the standard normal distribution. 
We would expect the values of the moments to be approximately close to those of the standard normal distribution.

In [32]:
# Standardised residuals within each regime
apple['Std_Resid'] = np.nan
for k in range(2):
    mask = apple['Regime'] == k
    apple.loc[mask, 'Std_Resid'] = (apple.loc[mask, 'log_returns'] - means[k]) / vols[k]

resid = apple['Std_Resid'].dropna()
x_norm = np.linspace(-4, 4, 200)

# QQ of standardised residuals
(osm_r, osr_r), (slope_r, intercept_r, _) = stats.probplot(resid, dist='norm')
qq_rx = np.array([osm_r[0], osm_r[-1]])
qq_ry = intercept_r + slope_r * qq_rx

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['QQ Plot of Standardised Residuals',
                    'Standardised Residuals vs N(0,1)'])

fig.add_trace(go.Scatter(x=osm_r, y=osr_r, mode='markers',
    marker=dict(color='steelblue', size=3), name='Residuals'), row=1, col=1)
fig.add_trace(go.Scatter(x=qq_rx, y=qq_ry,
    line=dict(color='red'), name='QQ line'), row=1, col=1)

fig.add_trace(go.Histogram(x=resid, nbinsx=60, histnorm='probability density',
    marker_color='steelblue', opacity=0.6, name='Residuals'), row=1, col=2)
fig.add_trace(go.Scatter(x=x_norm, y=stats.norm.pdf(x_norm),
    line=dict(color='red', width=2), name='N(0,1)'), row=1, col=2)

fig.update_xaxes(title_text='Theoretical Quantiles', row=1, col=1)
fig.update_yaxes(title_text='Sample Quantiles', row=1, col=1)
fig.update_xaxes(title_text='Standardised Residual', row=1, col=2)
fig.update_yaxes(title_text='Density', row=1, col=2)
fig.update_layout(height=450, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()

Both the QQ plot and the plot of the Standardized Residuals show that the data is closely normal so our model is a fairly good fit to the data.

In [33]:
# Skewness, kurtosis and Jarque-Bera test within each regime
print('Residual statistics within each regime:')
for k, lbl in enumerate(labels):
    resids = apple[apple['Regime'] == k]['Std_Resid'].dropna()
    print(f'  {lbl}:')
    print(f'    n = {len(resids)},  skewness = {resids.skew():.4f},  excess kurtosis = {resids.kurtosis():.4f}')
    jb, pval = stats.jarque_bera(resids)
    print(f'    Jarque-Bera: stat={jb:.2f}, p={pval:.4f}')

Residual statistics within each regime:
  Bull (low volatility):
    n = 1813,  skewness = -0.1001,  excess kurtosis = 0.6492
    Jarque-Bera: stat=34.36, p=0.0000
  Bear (high volatility):
    n = 196,  skewness = 0.2324,  excess kurtosis = 0.3233
    Jarque-Bera: stat=2.40, p=0.3014


Both values of the excess kurtosis are close to 0 so it indicates also that the model is fairly a good  fit to the data

---
## 6. Damage — Problems the Model Reveals

Key issues highlighted: 
1. ** underestimation of risks - The HMM model assumes that the returns follow a Gaussian (normal) distribution. The heavy tails, and excess kurtosis indicate that the data is not truly normal therefore it can underestimate the risk in times of crisis.


2. **Sensitivity to Model mis-specifications** - the model is  highly sensitive to how it is set up, number of states and variables. Also the assumptions behind the model such as stationarity.In this case picking  2 states might have been oversimplification.


3. The model takes time to detect change in state. 

---
## 7. Directions — Can We Improve the Model?
 Ways to improve the model 
1. **Increasing the number of states** - a state can be used to identify the state that is found in recovery (the period between the bullish and bearish regime.)

2. Using techniques like PCA or scaling to ensure that the data is more normal (to take care of the heavy tails.)

3. Training the model on shorter periods of time and retraining the model more frequently. 

4. Using HMM along with other models such as Random Forest to improve efficiency. 

---
## 8. Deployment — How To Implement the Model?

There can be several ways that can be used to implement a model like the HMM. 

1. **Guide trades and investments** - Trading when the regime predicts a "favorable regime". For instance, a long term strategy can be disabled when the probability of a high volatility/ bear state is high (more than 50%)

2. **Alongside specialist models ** - Train separate algorithms like Random Forest for each regime. When the HMM signals a regime shift, you switch to the model specifically optimized for those conditions.

3. **Asset allocation** - adjust the investment portfolio based on the predicted probability of each state. Eg. Moving to more capital into Bonds when the probability for a turbulent regime is high.



---
## Non-Technical Report

#### Section 1 - What I Found

After analysing Apple's daily stock returns from 2018 to 2025, the stock's behaviour clearly behaves as follows : a calm phase which prevailed most of the time (2018, most of 2019 and 2021, 2023–2024) and a turbulent phase (early 2020 and most of 2022 and 2025)

- **Calm Phase:** - this is seen through the more stable (low volatility) market trend with a growth in the prices of the stock. This trend was observed for most of the time. 
- **Turbulent Phase:** this is when the market was in an unstable trend (high volatility), there was also a decrease in the prices of the stock. This was observed around 2020, 2022 and 2025. 

#### Section 2 - Recommended Course of Action

- **During turbulent phases:** investment should be more focused on more stable assets that are not highly volatile. Such assets include property and bonds. Stocks such as AAPL should be avoided. 
- **During calm phases:** when the market is in a favorable trend then there is need to add more positions in order to fully capitalize on the favorable trend. Being careful to note any indication of a change. 
- Markets should be continuously monitored and there is need to invest in improvement of the prediction systems such as investment into AI incorporation.
#### Section 3 - What Drove the Phases

- **Calm phases:** The Apple products were utilising most of the market and there was evident growth. There were favorable financial and macro-economic policies in place. 
- **Turbulent phases:** These were due to sudden changes in policies (such as in 2025 when there was a change of leadership in the USA), pandemics (COVID-19) and general increase in inflation rates and interst rates. 

Thus, micro-economics policies can be considered to be a greater cause for the market behavior of Apple stock as well as good internal governancy policies.

---
### References

- Hamilton, J.D. A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. *Econometrica*, (1989).

- Rabiner, L.R. A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition. *Proceedings of the IEEE*, (1989).

- Hull, J.C.  *Options, Futures, and Other Derivatives* (10th ed.). Pearson. (2018).
-Wang, Linxi, et al. “Early Warning of Regime Switching in a Financial Time Series: A Heteroskedastic Network Model.” PLOS ONE, vol. 20, no. 10, 2025

- hmmlearn documentation: https://hmmlearn.readthedocs.io

- Yahoo Finance data accessed via yfinance Python library.